In [ ]:
from pathlib import Path

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm


In [8]:
MASKS_DIR = Path("Z:/nas-ctm01/datasets/public/LungNodule-nifti/preprocessed/MAISI/results_synth_maskct/masks_132_without_tumor")
LUNG_LOBE_LABELS = [28, 29, 30, 31, 32]
ANATOMY_LABELS = {
    28: "Left Upper Lobe",
    29: "Left Lower Lobe",
    30: "Right Upper Lobe",
    31: "Right Middle Lobe",
    32: "Right Lower Lobe",
}

In [9]:
mask_paths = sorted(path for path in MASKS_DIR.iterdir()
                        if path.is_file() and path.name.lower().endswith((".nii", ".nii.gz")))
print(f"Found {len(mask_paths)} mask files in {MASKS_DIR}")

Found 1036 mask files in Z:\nas-ctm01\datasets\public\LungNodule-nifti\preprocessed\MAISI\results_synth_maskct\masks_132_without_tumor


In [10]:
lobe_proportions = np.zeros([len(mask_paths), len(LUNG_LOBE_LABELS)])

for i, mask_path in enumerate(tqdm(mask_paths, desc="Processing masks")):
    mask = nib.load(mask_path).get_fdata()

    counts = np.array([np.count_nonzero(mask == label)
                       for label in [28, 29, 30, 31, 32]])

    left_lung_voxels = counts[:2].sum()
    right_lung_voxels = counts[2:].sum()

    denominators = np.array(
        [left_lung_voxels] * 2 + [right_lung_voxels] * 3
    )

    lobe_proportions[i] = np.divide(
        counts,
        denominators,
        out=np.full(5, np.nan),
        where=denominators != 0,
    )

    #print(f"Mask: {mask_path.name}: Lobe proportions: {lobe_proportions[i]}\n")

mean_proportions = np.nanmean(lobe_proportions, axis=0)

print(f"\n{'Lobe':<22} {'Mean proportion':>16}")
print("-" * 39)

for label, mean in zip([28, 29, 30, 31, 32], mean_proportions):
    print(f"{ANATOMY_LABELS[label]:<22} {mean:>15.2%}")


Processing masks: 100%|██████████| 1036/1036 [03:36<00:00,  4.78it/s]


Lobe                    Mean proportion
---------------------------------------
Left Upper Lobe                 57.24%
Left Lower Lobe                 42.76%
Right Upper Lobe                41.40%
Right Middle Lobe               16.63%
Right Lower Lobe                41.98%
